# EMO-X in 3 minutes (runs free on Colab, no API key)

Execution-based evaluation for AI coding models. This notebook runs the
frozen `code25` suite with the deterministic stub backend and charts
per-family results. Swap in any OpenAI-compatible endpoint for real models.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SE-Emam/EMO-X-Adaptive-Agent-Evaluation/blob/main/notebooks/emo_x_quickstart.ipynb)

In [ ]:
!git clone -q https://github.com/SE-Emam/EMO-X-Adaptive-Agent-Evaluation
%cd EMO-X-Adaptive-Agent-Evaluation

In [ ]:
# 0. Harness check — must end with RESULT: PASS
!python3 shared/run.py --self-test 2>&1 | tail -5

In [ ]:
# 1. Run the frozen code25 suite (stub backend = deterministic smoke)
# For a real model: --backend openai-generic --base-url ... --model ...
# Localhost endpoints need EMOX_ALLOW_LOCAL=1 (loopback guard):
#   EMOX_ALLOW_LOCAL=1 python3 shared/run.py --backend openai-generic --base-url http://localhost:11434/v1 --model ...
!python3 shared/run.py --backend stub --suite code25 --trials 1 --out results/ 2>&1 | tail -4

In [ ]:
# 2. Chart per-family pass rate from the sealed raw bundle
import glob, json, os
import matplotlib.pyplot as plt

rundir = sorted(glob.glob('results/raw/RUN-code25*'))[-1]
fams = {}
for line in open(os.path.join(rundir, 'events.jsonl')):
    e = json.loads(line)
    if e.get('primary_status') in ('PASS', 'FAIL'):
        fams.setdefault(e['task_family_id'], []).append(
            1 if e['primary_status'] == 'PASS' else 0)
names = sorted(fams)
rates = [sum(fams[n]) / len(fams[n]) for n in names]
plt.figure(figsize=(10, 3))
plt.bar(names, rates)
plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha='right')
plt.title('EMO-X code25 per-family pass rate (stub backend)')
plt.tight_layout()
plt.show()
print('bundle:', rundir)